In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer

import numpy as np
import pandas as pd
import time
import os
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
class AdditionalCustomDataset(Dataset):
    """
    Pre-tokenizes all data during initialization for much faster training.
    """
    def __init__(self, texts, labels, additional_texts, tokenizer, bert_tokenizer, max_length):
        self.labels = labels
        self.max_length = max_length
        
        # Pre-tokenize ALL data once during initialization (MUCH faster than tokenizing in __getitem__)
        print("Pre-tokenizing primary texts...")
        primary_encodings = tokenizer(
            texts,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        self.primary_input_ids = primary_encodings['input_ids']
        self.primary_attention_mask = primary_encodings['attention_mask']
        
        print("Pre-tokenizing additional texts...")
        additional_encodings = bert_tokenizer(
            additional_texts,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        self.additional_input_ids = additional_encodings['input_ids']
        self.additional_attention_mask = additional_encodings['attention_mask']
        
        print(f"Pre-tokenization complete. Dataset size: {len(self.labels)}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.primary_input_ids[idx],
            self.primary_attention_mask[idx],
            self.additional_input_ids[idx],
            self.additional_attention_mask[idx],
            self.labels[idx]
        )

In [ ]:
class ProjectionMLP(nn.Module):
        def __init__(self, input_size, output_size):
            super(ProjectionMLP, self).__init__()
            self.layers = nn.Sequential(
                nn.Linear(input_size, output_size),
                nn.ReLU(),
                nn.Linear(output_size, 2)
            )

        def forward(self, x):
            return self.layers(x)

In [ ]:
class GumbelTokenSelector(nn.Module):
        def __init__(self, hidden_size, tau=1.0):
            super().__init__()
            self.tau = tau
            self.proj = nn.Linear(hidden_size * 2, 1)
    
        def forward(self, token_embeddings, cls_embedding, training=True):
            """
            token_embeddings: (B, L, H)
            cls_embedding:   (B, H)
            """
            B, L, H = token_embeddings.size()
    
            cls_exp = cls_embedding.unsqueeze(1).expand(-1, L, -1)
            x = torch.cat([token_embeddings, cls_exp], dim=-1)
    
            logits = self.proj(x).squeeze(-1)  # (B, L)
    
            if training:
                probs = F.gumbel_softmax(
                    torch.stack([logits, torch.zeros_like(logits)], dim=-1),
                    tau=self.tau,
                    hard=False
                )[..., 0]
            else:
                probs = torch.sigmoid(logits)
    
            return probs, logits

In [ ]:
class MultiScaleAttentionCNN(nn.Module):
        def __init__(
            self,
            hidden_size=768,
            num_filters=128,
            kernel_sizes=(2, 3, 4),
            dropout=0.3,
        ):
            super().__init__()
    
            self.convs = nn.ModuleList([
                nn.Conv1d(hidden_size, num_filters, k)
                for k in kernel_sizes
            ])
    
            self.attention_fc = nn.Linear(num_filters, 1)
            self.dropout = nn.Dropout(dropout)
            self.out_dim = num_filters * len(kernel_sizes)
    
        def forward(self, x, mask):
            """
            x:    (B, L, H)
            mask: (B, L)
            """
            x = x.transpose(1, 2)  # (B, H, L)
            feats = []
    
            for conv in self.convs:
                h = F.relu(conv(x))           # (B, C, L')
                h = h.transpose(1, 2)         # (B, L', C)
    
                attn = self.attention_fc(h).squeeze(-1)
                attn = attn.masked_fill(mask[:, :attn.size(1)] == 0, -1e9)
                alpha = F.softmax(attn, dim=1)
    
                pooled = torch.sum(h * alpha.unsqueeze(-1), dim=1)
                feats.append(pooled)
    
            out = torch.cat(feats, dim=1)
            return self.dropout(out)

In [ ]:
class TemporalCNN(nn.Module):
        def __init__(
            self,
            hidden_size=768,
            num_filters=128,
            kernel_sizes=(2, 3, 4),
            dropout=0.1,
            dilation_base=2,
        ):
            super().__init__()
            
            self.kernel_sizes = kernel_sizes
            self.dilation_base = dilation_base
            
            # Dilated convolutions with exponentially increasing dilation rates
            self.convs = nn.ModuleList([
                nn.Conv1d(
                    hidden_size, 
                    num_filters, 
                    k,
                    dilation=dilation_base ** i,  # dilation = 2^i
                    padding=0  # we'll handle causal padding manually
                )
                for i, k in enumerate(kernel_sizes)
            ])
            
            self.dropout = nn.Dropout(dropout)
            self.out_dim = num_filters * len(kernel_sizes)
    
        def _causal_padding(self, x, kernel_size, dilation):
            """
            Apply left padding only (causal) to ensure output at time t
            only depends on inputs from time 0 to t.
            """
            # Calculate required padding: (kernel_size - 1) * dilation
            padding = (kernel_size - 1) * dilation
            # Pad only on the left (temporal dimension)
            return F.pad(x, (padding, 0))
    
        def forward(self, x, attention_mask):
            """
            x: (B, L, H)
            attention_mask: (B, L)
            """
            # zero-out padding tokens
            mask = attention_mask.unsqueeze(-1)
            x = x * mask
            
            # (B, H, L) for Conv1d
            x = x.transpose(1, 2)
            
            feats = []
            for i, conv in enumerate(self.convs):
                kernel_size = self.kernel_sizes[i]
                dilation = self.dilation_base ** i
                
                # Apply causal padding (left-only)
                x_padded = self._causal_padding(x, kernel_size, dilation)
                
                # Apply dilated convolution
                c = F.relu(conv(x_padded))  # (B, C, L')
                
                # Global max pooling over the temporal dimension
                p = F.max_pool1d(c, kernel_size=c.size(2)).squeeze(2)  # (B, C)
                
                feats.append(p)
            
            out = torch.cat(feats, dim=1)  # (B, num_filters * len(kernel_sizes))
            return self.dropout(out)

In [ ]:
class ConcatModel(nn.Module):
        def __init__(
            self,
            hatebert_model,
            additional_model,
            temporal_cnn,
            msa_cnn,
            selector,
            projection_mlp,
            unfreeze_n_layers_hate=12, #hatebert unfreeze all 12 layers
            unfreeze_n_layers_add=0, # additional bert freeze all layers
            hate_pooler=True,
            add_pooler=False
            
        ):
            super().__init__()
    
            self.hatebert_model = hatebert_model
            self.additional_model = additional_model
    
            self.temporal_cnn = temporal_cnn
            self.msa_cnn = msa_cnn
            self.selector = selector
            self.projection_mlp = projection_mlp
    
            # freeze everything on additional bert
            for p in self.additional_model.parameters():
                p.requires_grad = False

            # will unfreeze last n layers of additional model
            add_num_layers = len(self.additional_model.encoder.layer)
            for i in range(add_num_layers - unfreeze_n_layers_add, add_num_layers):
                for param in self.additional_model.encoder.layer[i].parameters():
                    param.requires_grad = True

            if self.additional_model.pooler is not None and add_pooler:
                for p in self.additional_model.pooler.parameters():
                    p.requires_grad = True

            # freeze everything on hatebert
            for p in self.hatebert_model.parameters():
                p.requires_grad = False
                
            # FIX: Use hatebert_model to get the correct number of layers
            hate_num_layers = len(self.hatebert_model.encoder.layer)
            for i in range(hate_num_layers - unfreeze_n_layers_hate, hate_num_layers):
                for param in self.hatebert_model.encoder.layer[i].parameters():
                    param.requires_grad = True
                    
            if self.hatebert_model.pooler is not None and hate_pooler:
                for p in self.hatebert_model.pooler.parameters():
                    p.requires_grad = True

    
        def forward(self, input_ids, attention_mask, additional_input_ids, additional_attention_mask):
            # ================= HateBERT =================
            hate_outputs = self.hatebert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            seq_emb = hate_outputs.last_hidden_state      # (B, L, H)
            cls_emb = seq_emb[:, 0, :]                    # (B, H)
            
            # ---- Token Selector ----
            token_probs, token_logits = self.selector(seq_emb, cls_emb, self.training)
            
            # ---- Temporal CNN on FULL embeddings (NOT masked) ----
            temporal_feat = self.temporal_cnn(seq_emb, attention_mask)
            
            # ---- Rationale-Weighted Summary Vector H_r ----
            weights = token_probs.unsqueeze(-1)  # (B, L, 1)
            H_r = (seq_emb * weights).sum(dim=1) / (weights.sum(dim=1) + 1e-6)
            
            # ================= Frozen Rationale BERT =================
            add_outputs = self.additional_model(
                input_ids=additional_input_ids,
                attention_mask=additional_attention_mask,
            )
            add_seq = add_outputs.last_hidden_state
            
            # ---- Multi-Scale Attention CNN ----
            msa_feat = self.msa_cnn(add_seq, additional_attention_mask)
            
            # ================= CONCAT (4 components) =================
            concat = torch.cat([cls_emb, temporal_feat, msa_feat, H_r], dim=1)
            
            logits = self.projection_mlp(concat)
            return logits

In [ ]:
class EarlyStopping:
    """
    Early stopping to stop training when validation loss doesn't improve.
    """
    def __init__(self, patience=10, min_delta=1.0, mode='min', verbose=True):
        """
        Args:
            patience (int): How many epochs to wait after last improvement.
            min_delta (float): Minimum change to qualify as an improvement.
            mode (str): 'min' for loss, 'max' for accuracy/f1.
            verbose (bool): Print messages when improvement occurs.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, current_score, model):
        """
        Call this after each epoch with the validation metric and model.
        
        Args:
            current_score: Current epoch's validation metric (loss, accuracy, f1, etc.)
            model: The model to save if there's improvement
            
        Returns:
            bool: True if training should stop, False otherwise
        """
        if self.best_score is None:
            # First epoch
            self.best_score = current_score
            self.save_checkpoint(model)
            if self.verbose:
                print(f"Initial best score: {self.best_score:.4f}")
        else:
            # Check if there's improvement
            if self.mode == 'min':
                improved = current_score < (self.best_score - self.min_delta)
            else:  # mode == 'max'
                improved = current_score > (self.best_score + self.min_delta)
            
            if improved:
                self.best_score = current_score
                self.save_checkpoint(model)
                self.counter = 0
                if self.verbose:
                    print(f"Validation improved! New best score: {self.best_score:.4f}")
            else:
                self.counter += 1
                if self.verbose:
                    print(f"No improvement. Patience counter: {self.counter}/{self.patience}")
                
                if self.counter >= self.patience:
                    self.early_stop = True
                    if self.verbose:
                        print(f"Early stopping triggered! Best score: {self.best_score:.4f}")
        
        return self.early_stop
    
    def save_checkpoint(self, model):
        """Save model state dict"""
        import copy
        self.best_model_state = copy.deepcopy(model.state_dict())
    
    def load_best_model(self, model):
        """Load the best model state into the model"""
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)
            if self.verbose:
                print(f"Loaded best model with score: {self.best_score:.4f}")
        return model

In [ ]:
def main(args):
    torch.manual_seed(args.seed)
    torch.cuda.empty_cache()
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


    file_map = {
            "gab": '/kaggle/input/datasets/jonniellm/final-dataset/Mistral_Rationales_file_GAB_dataset(85-15).csv',
            "twitter": '/kaggle/input/datasets/jonniellm/final-dataset/Mistral_Rationales_file_Twitter_dataset(85-15).csv',
            "reddit": '/kaggle/input/datasets/jonniellm/final-dataset/Mistral_Rationales_file_REDDIT_dataset(85-15).csv',
            "youtube": '/kaggle/input/datasets/jonniellm/final-dataset/Mistral_Rationales_file_YOUTUBE_dataset(85-15).csv',
            "implicit": '/kaggle/input/datasets/jonniellm/final-dataset/Mistral_Rationales_file_IMPLICIT_dataset(85-15).csv'
        }

    file_path = file_map[args.dataset]
    df = pd.read_csv(file_path)
    train_df = df[df['exp_split'] == 'train']
    test_df = df[df['exp_split'] == 'test']

    print("Train df: ", len(train_df))
    print("Test_df: ", len(test_df))

    import gc
    # del variables
    gc.collect()

    
    tokenizer = args.hate_tokenizer ## need this for tokenizing the input text in data loader
    tokenizer_bert = args.bert_tokenizer
    #Splitting training and validation testing split to test accuracy
    train_idx, val_idx = train_test_split(
        train_df.index,
        test_size=0.2,
        stratify=train_df["label"],
        random_state=args.seed
    )
    
    if args.dataset == "implicit":
        train_text = train_df.loc[train_idx, "post"].tolist()
        val_texts  = train_df.loc[val_idx, "post"].tolist()
    else:
        train_text = train_df.loc[train_idx, "text"].tolist()
        val_texts  = train_df.loc[val_idx, "text"].tolist()
    
    add_train_text = train_df.loc[train_idx, "Mistral_Rationales"].tolist()
    add_val_texts  = train_df.loc[val_idx, "Mistral_Rationales"].tolist()
    
    train_labels = train_df.loc[train_idx, "label"].tolist()
    val_labels   = train_df.loc[val_idx, "label"].tolist()

    train_dataset = AdditionalCustomDataset(
        train_text,
        train_labels,
        add_train_text,
        tokenizer,
        tokenizer_bert,
        max_length=512
    )
    
    val_dataset = AdditionalCustomDataset(
        val_texts,
        val_labels,
        add_val_texts,
        tokenizer,
        tokenizer_bert,
        max_length=512
    )

    #Creating dataloader object to train the model
    # num_workers=2 for parallel data loading, pin_memory=True for faster GPU transfer
    train_dataloader = DataLoader(
        train_dataset, 
        batch_size=args.batch_size, 
        shuffle=True,
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )
    val_dataloader = DataLoader(
        val_dataset, 
        batch_size=args.batch_size, 
        shuffle=False,
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )
    hatebert_model = args.hatebert_model
    additional_model = args.additional_model
    
            
    temporal_cnn = TemporalCNN(
        hidden_size=768,
        num_filters=args.temp_filters, # 64 - 128
        kernel_sizes=(2, 3, 4),
        dropout=args.temp_dropout,
        dilation_base=args.temp_dilate # 0 - 4
    ).to(device)

    msa_cnn = MultiScaleAttentionCNN(
        hidden_size=768,
        num_filters=args.msa_filters, # 64 - 128
        kernel_sizes=(2, 3, 4),
        dropout=args.msa_dropout
    ).to(device)
    
    selector = GumbelTokenSelector(
        hidden_size=768,
        tau=1.0
    ).to(device)
    
    projection_mlp = ProjectionMLP(
        input_size=temporal_cnn.out_dim + msa_cnn.out_dim + 768 * 2,
        output_size=512
    ).to(device)



    concat_model = ConcatModel(
        hatebert_model=hatebert_model,
        additional_model=additional_model,
        temporal_cnn=temporal_cnn,
        msa_cnn=msa_cnn,
        selector=selector,
        projection_mlp=projection_mlp,
        unfreeze_n_layers_hate=args.hate_layers, #hatebert unfreeze all 12 layers id 12 by default
        unfreeze_n_layers_add=args.add_layers, # additional bert freeze all layers if 0 by default
        hate_pooler=args.hate_pooler, #bool to controle if pooler is frozen or not true=not frozen
        add_pooler=args.add_pooler #bool to controle if pooler is frozen or not true=not froze
    ).to(device)

    optimizer = AdamW(
        concat_model.parameters(),
        lr=args.lr, # 2e-5
        weight_decay=args.wd
    )
    criterion = nn.CrossEntropyLoss().to(device)

    # criterion = criterion.to(device)

    os.makedirs("/kaggle/working/models", exist_ok=True)

    history = {
        "train_loss": [],
        "val_loss": [], #loss
        "train_acc": [],
        "train_precision": [],
        "train_recall": [],
        "train_f1": [], #train binary
        "val_acc": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1": [], #validation binary
        "train_acc_weighted": [],
        "train_precision_weighted": [],
        "train_recall_weighted": [],
        "train_f1_weighted": [], # train weighted
        "val_acc_weighted": [],
        "val_precision_weighted": [],
        "val_recall_weighted": [],
        "val_f1_weighted": [], # validation weighted
        "train_acc_macro": [],
        "train_precision_macro": [],
        "train_recall_macro": [],
        "train_f1_macro": [], #train macro
        "val_acc_macro": [],
        "val_precision_macro": [],
        "val_recall_macro": [],
        "val_f1_macro": [],# validation macro
        "epoch_time": [],
        "train_throughput": [],
        "val_confidence_mean": [],
        "val_confidence_std": [],
        "gpu_memory_mb": [],
    }

    early_stopping = EarlyStopping(patience=args.patience, min_delta=0.001, mode='min', verbose=True) # early stop on loss

    for epoch in range(args.num_epochs):
        epoch_val_confidences = []
        epoch_start_time = time.time()
        samples_seen = 0
        
        concat_model.train()

        train_losses = []
        train_preds = []
        train_labels_epoch = []
        train_accuracy = 0
        train_epoch_size = 0

        with tqdm(train_dataloader, desc=f'Epoch {epoch + 1}', dynamic_ncols=True) as loop:
            for batch in loop:
                input_ids, attention_mask, additional_input_ids, additional_attention_mask, labels = batch
                
                samples_seen += labels.size(0)
                
                if torch.cuda.is_available():
                    input_ids = input_ids.to(device)
                    attention_mask = attention_mask.to(device)
                    additional_input_ids = additional_input_ids.to(device)
                    additional_attention_mask = additional_attention_mask.to(device)
                    labels = labels.to(device)

                # Forward pass through the ConcatModel
                optimizer.zero_grad()
                outputs = concat_model(input_ids=input_ids, attention_mask=attention_mask, additional_input_ids=additional_input_ids, additional_attention_mask=additional_attention_mask)
                loss = criterion(outputs, labels)

                # Backward pass and optimization
                loss.backward()
                torch.nn.utils.clip_grad_norm_(concat_model.parameters(), max_norm=args.max_grad_norm)
                optimizer.step()
                
                probs = torch.softmax(outputs, dim=1)
                confidences, predictions = torch.max(probs, dim=1)
                train_preds.extend(predictions.cpu().numpy())
                train_labels_epoch.extend(labels.cpu().numpy())

                train_losses.append(loss.item())

                # Update accuracy and epoch size
                predictions = torch.argmax(outputs, dim=1)
                train_accuracy += (predictions == labels).sum().item()
                train_epoch_size += len(labels)
                
        epoch_train_time = time.time() - epoch_start_time
        train_throughput = samples_seen / epoch_train_time       
    
        # Calculate train metrics (binary, weighted, macro)
        train_precision, train_recall, train_f1, _ = precision_recall_fscore_support(
            train_labels_epoch, train_preds, average='binary'
        )
        train_precision_weighted, train_recall_weighted, train_f1_weighted, _ = precision_recall_fscore_support(
            train_labels_epoch, train_preds, average='weighted'
        )
        train_precision_macro, train_recall_macro, train_f1_macro, _ = precision_recall_fscore_support(
            train_labels_epoch, train_preds, average='macro'
        )
        train_acc = accuracy_score(train_labels_epoch, train_preds)

        # Evaluation on the validation set
        concat_model.eval()

        val_predictions = []
        val_labels_epoch = []
        val_loss = 0
        num_batches = 0

        with torch.no_grad(), tqdm(val_dataloader, desc='Validation', dynamic_ncols=True) as loop:
            for batch in loop:
                input_ids, attention_mask, additional_input_ids, additional_attention_mask, labels = batch

                if torch.cuda.is_available():
                    input_ids = input_ids.to(device)
                    attention_mask = attention_mask.to(device)
                    additional_input_ids = additional_input_ids.to(device)
                    additional_attention_mask = additional_attention_mask.to(device)
                    labels = labels.to(device)

                # Forward pass through the ConcatModel
                outputs = concat_model(input_ids=input_ids, attention_mask=attention_mask, additional_input_ids=additional_input_ids, additional_attention_mask=additional_attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                num_batches += 1
                probs = torch.softmax(outputs, dim=1)
                confidences, predictions = torch.max(probs, dim=1)

                epoch_val_confidences.extend(confidences.cpu().numpy())
                
                val_predictions.extend(predictions.cpu().numpy())
                val_labels_epoch.extend(labels.cpu().numpy())
        
        val_loss /= num_batches
        
        # Calculate validation metrics (binary)
        val_accuracy = accuracy_score(val_labels_epoch, val_predictions)
        val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(
            val_labels_epoch, val_predictions, average='binary'
        )
        # Calculate validation metrics (weighted)
        val_precision_weighted, val_recall_weighted, val_f1_weighted, _ = precision_recall_fscore_support(
            val_labels_epoch, val_predictions, average='weighted'
        )
        # Calculate validation metrics (macro)
        val_precision_macro, val_recall_macro, val_f1_macro, _ = precision_recall_fscore_support(
            val_labels_epoch, val_predictions, average='macro'
        )
        
        print(f"Epoch {epoch}:")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Validation Accuracy: {val_accuracy:.4f}")
        print(f"  Train Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}")
        print(f"  Val Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
        print(f"  Avg. Train Loss: {sum(train_losses) / len(train_losses):.4f}")
        print(f"  Validation Loss: {val_loss:.4f}")
        epoch_time = time.time() - epoch_start_time
        conf_mean = np.mean(epoch_val_confidences)
        conf_std = np.std(epoch_val_confidences)

        # Append all train metrics to history
        history["train_loss"].append(np.mean(train_losses))
        history["train_acc"].append(train_acc)
        history["train_precision"].append(train_precision)
        history["train_recall"].append(train_recall)
        history["train_f1"].append(train_f1)
        history["train_acc_weighted"].append(train_acc)  # accuracy is same for all averaging
        history["train_precision_weighted"].append(train_precision_weighted)
        history["train_recall_weighted"].append(train_recall_weighted)
        history["train_f1_weighted"].append(train_f1_weighted)
        history["train_acc_macro"].append(train_acc)  # accuracy is same for all averaging
        history["train_precision_macro"].append(train_precision_macro)
        history["train_recall_macro"].append(train_recall_macro)
        history["train_f1_macro"].append(train_f1_macro)
        
        # Append all validation metrics to history
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_accuracy)
        history["val_precision"].append(val_precision)
        history["val_recall"].append(val_recall)
        history["val_f1"].append(val_f1)
        history["val_acc_weighted"].append(val_accuracy)  # accuracy is same for all averaging
        history["val_precision_weighted"].append(val_precision_weighted)
        history["val_recall_weighted"].append(val_recall_weighted)
        history["val_f1_weighted"].append(val_f1_weighted)
        history["val_acc_macro"].append(val_accuracy)  # accuracy is same for all averaging
        history["val_precision_macro"].append(val_precision_macro)
        history["val_recall_macro"].append(val_recall_macro)
        history["val_f1_macro"].append(val_f1_macro)
        
        # Append efficiency metrics
        history["epoch_time"].append(epoch_time)
        history["train_throughput"].append(train_throughput)
        history["val_confidence_mean"].append(conf_mean)
        history["val_confidence_std"].append(conf_std)
        
        if torch.cuda.is_available():
            history["gpu_memory_mb"].append(
                torch.cuda.max_memory_allocated() / 1024**2
            )
            torch.cuda.reset_peak_memory_stats()
        else:
            history["gpu_memory_mb"].append(0)
        
        print(f"  Epoch Time (s): {epoch_time:.2f}")
        print(f"  Throughput (samples/sec): {train_throughput:.2f}")
        print(f"  Val Confidence Mean: {conf_mean:.4f} ± {conf_std:.4f}")
        
        current_metric = val_loss
        if early_stopping(current_metric, concat_model):
                print(f"\n{'='*50}")
                print(f"Early stopping at epoch {epoch+1}")
                print(f"{'='*50}\n")
                break
    model = early_stopping.load_best_model(concat_model)
    torch.save(model.state_dict(), f"/kaggle/working/models/{args.dataset}_concat_model.pt")
    print(f"Best model saved to /kaggle/working/models/{args.dataset}_concat_model.pt")

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "history": history,
    }
    torch.save(checkpoint, f"/kaggle/working/models/{args.dataset}_concat_checkpoint.pt")
    print(f"Checkpoint with history saved to /kaggle/working/models/{args.dataset}_concat_checkpoint.pt")

    if args.dataset == "implicit":
        test_texts = test_df["post"].tolist()
    else:
        test_texts = test_df["text"].tolist()
    
    add_test_texts = test_df["Mistral_Rationales"].tolist()
    test_labels = test_df["label"].tolist()

    test_dataset = AdditionalCustomDataset(test_texts, test_labels, add_test_texts, tokenizer, tokenizer_bert, max_length=512)
    test_dataloader = DataLoader(
        test_dataset, 
        batch_size=args.batch_size,  # Use same batch size as training for faster inference
        shuffle=False,
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )

    # ================= TEST EVALUATION WITH EFFICIENCY =================
    model.eval()
    test_predictions = []
    test_true_labels = []
    test_confidences = []

    samples_seen = 0
    test_start_time = time.time()
    
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    with torch.no_grad(), tqdm(test_dataloader, desc='Testing', dynamic_ncols=True) as loop:
        for batch in loop:
            input_ids, attention_mask, additional_input_ids, additional_attention_mask, labels = batch
    
            batch_size = labels.size(0)
            samples_seen += batch_size
    
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            additional_input_ids = additional_input_ids.to(device)
            additional_attention_mask = additional_attention_mask.to(device)
            labels = labels.to(device)
    
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                additional_input_ids=additional_input_ids,
                additional_attention_mask=additional_attention_mask
            )
    
            probs = torch.softmax(outputs, dim=1)
            confidences, preds = torch.max(probs, dim=1)
    
            test_confidences.extend(confidences.cpu().numpy())
            test_predictions.extend(preds.cpu().numpy())
            test_true_labels.extend(labels.cpu().numpy())

    
    # ================= TEST METRICS =================
    test_time = time.time() - test_start_time
    test_throughput = samples_seen / test_time
    
    accuracy = accuracy_score(test_true_labels, test_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_true_labels, test_predictions, average='weighted'
    )
    conf_mean = np.mean(test_confidences)
    conf_std = np.std(test_confidences)
    cm = confusion_matrix(test_true_labels, test_predictions)
    
    gpu_memory_mb = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0

    print("\n================= FINAL TEST RESULTS =================")
    print(f"Dataset: {args.dataset}, Seed: {args.seed}, Epochs: {args.num_epochs}")
    print(f"Test Accuracy : {accuracy:.4f}")
    print(f"Test Precision: {precision:.4f}")
    print(f"Test Recall   : {recall:.4f}")
    print(f"Test F1-score : {f1:.4f}")
    print(f"Test Confidence Mean ± Std: {conf_mean:.4f} ± {conf_std:.4f}")
    print(f"Test Time (s) : {test_time:.2f}")
    print(f"Throughput (samples/sec) : {test_throughput:.2f}")
    print(f"Peak GPU Memory (MB) : {gpu_memory_mb:.2f}")
    print("\nClassification Report:")
    print(classification_report(test_true_labels, test_predictions))
    print("\nConfusion Matrix:")
    # Plot confusion matrix with seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Non-Hate', 'Hate'],
                yticklabels=['Non-Hate', 'Hate'],
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontsize=14, pad=20)
    plt.ylabel('True label', fontsize=12)
    plt.xlabel('Predicted label', fontsize=12)
    plt.tight_layout()
    plt.show()
    print("======================================================")

    return model, history

In [ ]:
from argparse import Namespace
from transformers import AutoTokenizer, AutoModel

# Load tokenizers and models ONCE (outside objective to save time)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
hate_tokenizer = AutoTokenizer.from_pretrained("GroNLP/hateBERT")
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

torch.cuda.empty_cache()

# Load fresh models for each trial (to reset weights)
hatebert_model = AutoModel.from_pretrained("GroNLP/hateBERT").to(device)
additional_model = AutoModel.from_pretrained("bert-base-uncased").to(device)

# Sample hyperparameters
args = Namespace(
    # Fixed
    seed=42,
    dataset="reddit",  # Change as needed: "gab", "twitter", "reddit", "youtube", "implicit"
    
    # Tokenizers & Models
    hate_tokenizer=hate_tokenizer,
    bert_tokenizer=bert_tokenizer,
    hatebert_model=hatebert_model,
    additional_model=additional_model,
    
    # Training hyperparameters
    batch_size=32,
    num_epochs=20,
    lr=1e-5,
    wd=0.01,
    patience=2,
    max_grad_norm=2.0,
    
    # TemporalCNN hyperparameters
    temp_filters=256,
    temp_dropout=0.1,
    temp_dilate=3,
    
    # MultiScaleAttentionCNN hyperparameters
    msa_filters=64,
    msa_dropout=0.23,
    
    # Layer unfreezing hyperparameters
    hate_layers=10,
    add_layers=0,
    hate_pooler=True,
    add_pooler=True,
)
    
model, history = main(args)


   

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 15))

# Row 1: Train vs Val comparisons
# Accuracy
axes[0, 0].plot(history['train_acc'], label='Train', marker='o')
axes[0, 0].plot(history['val_acc'], label='Validation', marker='s')
axes[0, 0].set_title('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Loss
axes[0, 1].plot(history['train_loss'], label='Train', marker='o')
axes[0, 1].plot(history['val_loss'], label='Validation', marker='s')
axes[0, 1].set_title('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

# F1 Score
axes[0, 2].plot(history['train_f1'], label='Train', marker='o')
axes[0, 2].plot(history['val_f1'], label='Validation', marker='s')
axes[0, 2].set_title('F1 Score (Binary)')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('F1')
axes[0, 2].legend()
axes[0, 2].grid(True)

# Precision
axes[0, 3].plot(history['train_precision'], label='Train', marker='o')
axes[0, 3].plot(history['val_precision'], label='Validation', marker='s')
axes[0, 3].set_title('Precision (Binary)')
axes[0, 3].set_xlabel('Epoch')
axes[0, 3].set_ylabel('Precision')
axes[0, 3].legend()
axes[0, 3].grid(True)

# Row 2: More Train vs Val + Individual metrics
# Recall
axes[1, 0].plot(history['train_recall'], label='Train', marker='o')
axes[1, 0].plot(history['val_recall'], label='Validation', marker='s')
axes[1, 0].set_title('Recall (Binary)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Recall')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Epoch Time
axes[1, 1].plot(history['epoch_time'], marker='o', color='green')
axes[1, 1].set_title('Epoch Time')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Time (s)')
axes[1, 1].grid(True)

# Train Throughput
axes[1, 2].plot(history['train_throughput'], marker='o', color='purple')
axes[1, 2].set_title('Train Throughput')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Samples/sec')
axes[1, 2].grid(True)

# Validation Confidence
axes[1, 3].errorbar(range(len(history['val_confidence_mean'])), 
                    history['val_confidence_mean'], 
                    yerr=history['val_confidence_std'], 
                    marker='o', color='orange', capsize=3)
axes[1, 3].set_title('Validation Confidence')
axes[1, 3].set_xlabel('Epoch')
axes[1, 3].set_ylabel('Confidence')
axes[1, 3].grid(True)

# Row 3: GPU Memory and Weighted/Macro metrics
# GPU Memory
axes[2, 0].plot(history['gpu_memory_mb'], marker='o', color='red')
axes[2, 0].set_title('GPU Memory Usage')
axes[2, 0].set_xlabel('Epoch')
axes[2, 0].set_ylabel('Memory (MB)')
axes[2, 0].grid(True)

# Weighted F1 comparison
axes[2, 1].plot(history['train_f1_weighted'], label='Train', marker='o')
axes[2, 1].plot(history['val_f1_weighted'], label='Validation', marker='s')
axes[2, 1].set_title('F1 Score (Weighted)')
axes[2, 1].set_xlabel('Epoch')
axes[2, 1].set_ylabel('F1')
axes[2, 1].legend()
axes[2, 1].grid(True)

# Macro F1 comparison
axes[2, 2].plot(history['train_f1_macro'], label='Train', marker='o')
axes[2, 2].plot(history['val_f1_macro'], label='Validation', marker='s')
axes[2, 2].set_title('F1 Score (Macro)')
axes[2, 2].set_xlabel('Epoch')
axes[2, 2].set_ylabel('F1')
axes[2, 2].legend()
axes[2, 2].grid(True)

# Hide last subplot
axes[2, 3].axis('off')

plt.suptitle('Training History', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()